# Scrapalot MoE Training Notebook — Qwen 3.6 35B-A3B on RunPod

**Goal**: Fine-tune **Qwen 3.6 35B-A3B** — Alibaba's open-weight Mixture-of-Experts model (35B total parameters, 3B active per token via expert routing) — on Scrapalot's per-collection Q&A JSONL datasets (produced by `scripts/dataset_generator/`) using Unsloth + LoRA.

**Why this base model**: Scrapalot's long-term vision is **expert-per-collection routing** — "anthropology expert", "AI expert", "philosophy expert" — where the gating layer picks which sub-expert handles a given query. Qwen 3.6 35B-A3B already ships with the MoE infrastructure we need (gating + experts already trained), so the work becomes *fine-tuning expert routing toward Scrapalot collections* instead of building a new MoE topology from scratch. A simpler intermediate step (per-collection LoRA adapters on top of the shared MoE base) is the practical starting point.

**Status**: SFT baseline (this notebook). Subsequent arxiv training papers iterate on this notebook (VPO, DPO, GRPO, ...) — see the *Contribution log* at the bottom.

**Strategic context**: see `memory/project_training_strategy.md` and `memory/feedback_training_papers_iterate_notebook.md`.

## RunPod setup

1. **GPU tier**: A100 40GB or larger is sufficient with 4-bit QLoRA (Qwen3.6-35B-A3B is MoE — only ~3B parameters active per token, base footprint quantized to 4-bit ≈ 22 GB + LoRA adapter ~1 GB + activations ~8 GB at seq=4096). For bf16 full precision use A100 80GB or H100 80GB. Avoid sub-24GB cards.
2. **Pod image**: `runpod/pytorch:2.4.0-py3.11-cuda12.4.1-devel-ubuntu22.04` (or newer PyTorch image with CUDA 12.4+).
3. **Volume**: 150 GB persistent volume mounted at `/workspace` — the unquantized 35B safetensors take ~70 GB + dataset + checkpoints + GGUF export. 4-bit-only setup can fit in 100 GB.
4. **Network volume** (optional): if you have Scrapalot's `datasets/*.jsonl` outputs locally, upload via `runpodctl send` or sync from S3.
5. **Estimated cost**: A100 40GB at ~$1.20/hr on RunPod community cloud (cheaper than 80GB!) → SFT run for 500 steps on a 50k-pair dataset costs ~$6-12. The MoE's active-3B character makes wall-clock training much faster than equivalent 35B-dense fine-tuning.

## 1. Environment setup

Install Unsloth + its training dependencies. The `--force-reinstall --no-cache-dir` on unsloth is required because of frequent breaking changes; pin a known-good version in production runs.

In [ ]:
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
!pip install torch torchvision pillow transformers trl peft datasets accelerate bitsandbytes

import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "n/a")

## 2. Model loading (Qwen 3.6 35B-A3B — MoE)

Loading Alibaba's Qwen 3.6 35B-A3B (Mixture-of-Experts: 35 B total parameters, 3 B active per token). Unsloth hosts the 16-bit safetensor variant at `unsloth/Qwen3.6-35B-A3B` (verified 2026-05-22). Fall back to `unsloth/Qwen3.6-27B` (dense) on compute-constrained pods, or to `unsloth/Qwen3-14B` for proof-of-concept runs.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 4096  # Scrapalot books → long answers. Raise to 8192 if VRAM allows.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.6-35B-A3B",  # MoE, 35B total / 3B active. Swap to 'unsloth/Qwen3.6-27B' for dense fallback.
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,  # 4-bit QLoRA — fits MoE-35B on A100 40GB. Set False + load_in_16bit=True for bf16 on A100/H100 80GB.
    load_in_16bit=False,
    full_finetuning=False,
)

## 3. LoRA configuration

Standard target_modules covering attention + MLP. For Qwen3.6-35B-A3B (MoE), the `gate_proj`/`up_proj`/`down_proj` modules are **inside each expert** — Unsloth's MoE-aware LoRA handles this transparently (one LoRA pair per expert MLP). `r=16` is a reasonable default; raise to 32 or 64 if the dataset is large (>200k pairs) or when adding per-expert specialisation in later contribution iterations.

**MoE-specific note**: if you later want to fine-tune expert *routing* (not just expert content) — e.g. teach the gate to send Scrapalot-anthropology queries preferentially to specific experts — add the gating projections to `target_modules`. That's a research spike covered in the Contribution log below, not a default.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's checkpointing → ~30% extra VRAM savings.
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

## 4. Dataset loading — Scrapalot per-collection Q&A JSONL

The Scrapalot `scripts/dataset_generator/` pipeline emits one JSONL per collection (e.g. `datasets/ufo.jsonl`, `datasets/agriculture.jsonl`, ...). Each line is one `{question, answer, thinking, topics, quality_score}` record. For SFT we concatenate question+answer with the Qwen chat template; the `thinking` field becomes a `<think>` block.

**Filter on quality_score >= 4** to keep only high-quality pairs. The Claude Code Q&A generator already self-rates 1-5; keeping ≥4 typically retains ~60-70% of pairs.

In [ ]:
import glob
import json
import os

from datasets import Dataset

# Adjust path: copy datasets/*.jsonl into the RunPod pod's /workspace/datasets/ before running.
DATASET_DIR = "/workspace/datasets"
QUALITY_THRESHOLD = 4

# Hyperfitting subset (per arXiv:2605.22579, see #hyperfitting in Contribution log).
# When True, downsample to 2000 highest-quality samples and (in Section 5) switch trainer
# to long-epoch near-zero-loss recipe. Default False keeps the standard SFT path.
HYPERFIT_MODE = False
HYPERFIT_TARGET_SAMPLES = 2000

records = []
for jsonl_path in sorted(glob.glob(f"{DATASET_DIR}/*.jsonl")):
    collection = os.path.basename(jsonl_path).removesuffix(".jsonl")
    with open(jsonl_path) as f:
        for line in f:
            obj = json.loads(line)
            qs = obj.get("quality_score", 0)
            if qs < QUALITY_THRESHOLD:
                continue
            records.append(
                {
                    "collection": collection,
                    "question": obj["question"],
                    "answer": obj["answer"],
                    "thinking": obj.get("thinking", ""),
                    "topics": obj.get("topics", []),
                    "quality_score": qs,
                }
            )

print(f"Loaded {len(records)} records from {DATASET_DIR}/*.jsonl (quality >= {QUALITY_THRESHOLD})")

if HYPERFIT_MODE:
    # Paper recipe: 2000 samples, train to near-zero loss. Sort highest-quality-first
    # so that even when the corpus is large, the subset is the "best of the best".
    records.sort(key=lambda r: -r["quality_score"])
    records = records[:HYPERFIT_TARGET_SAMPLES]
    print(f"HYPERFIT_MODE on → kept top {len(records)} by quality_score (paper §2.1 recipe)")


def to_chat(example):
    messages = [
        {"role": "user", "content": example["question"]},
        {
            "role": "assistant",
            "content": (f"<think>{example['thinking']}</think>\n{example['answer']}" if example.get("thinking") else example["answer"]),
        },
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}


raw_ds = Dataset.from_list(records)
ds = raw_ds.map(to_chat, remove_columns=raw_ds.column_names)
split = ds.train_test_split(test_size=0.1, seed=3407)
train_ds = split["train"]
eval_ds = split["test"]
print(f"Train: {len(train_ds)}  Eval: {len(eval_ds)}")

## 5. SFT trainer setup

`max_steps=500` is a reasonable first pass. For a real production run scale by dataset size: aim for 2-3 epochs over the full training split.

In [ ]:
from trl import SFTConfig, SFTTrainer

# Two configs, selected by the HYPERFIT_MODE flag set in Section 4.
# - default: standard SFT, 500 steps, cosine LR decay, weight_decay=0.01.
# - hyperfit: paper recipe (arXiv:2605.22579) — drive loss to near-zero on the
#   2000-sample subset. 260 epochs at batch 4 ≈ 130k steps total; in practice
#   diversity plateaus much earlier (paper §6 Table 8: TTR doubles by epoch 20,
#   peaks 80-140). Configure for early stopping by watching loss<0.07.
if HYPERFIT_MODE:
    sft_args = SFTConfig(
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        # Paper §2.1: 260 epochs on |D|=2000 (no early stopping; weight decay = 0).
        # With batch=1 * grad_accum=4 → effective batch 4, ~500 steps per epoch on 2000.
        # Set num_train_epochs (overrides max_steps). Empirically epoch 80-140 is the
        # useful window — set epochs lower if you want a partial-recipe run.
        num_train_epochs=260,
        max_steps=-1,
        learning_rate=2e-4,
        logging_steps=10,
        eval_steps=500,
        save_steps=2000,
        output_dir="/workspace/checkpoints/qwen36_35b_a3b_hyperfit",
        optim="adamw_8bit",
        # CRITICAL paper §2.1: regularization is the antagonist here — set to zero.
        weight_decay=0.0,
        lr_scheduler_type="constant",  # paper has no cosine decay in the hyperfit setup
        seed=3407,
        dataset_num_proc=2,
        report_to="none",
    )
else:
    sft_args = SFTConfig(
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=500,
        learning_rate=2e-4,
        logging_steps=10,
        eval_steps=100,
        save_steps=250,
        output_dir="/workspace/checkpoints/qwen36_35b_a3b_sft",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        dataset_num_proc=2,
        report_to="none",
    )

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    args=sft_args,
)
print(f"Trainer ready. Mode = {'hyperfit (arXiv:2605.22579)' if HYPERFIT_MODE else 'default SFT'}")

## 6. Train

Expect ~3-5 hours on A100 80GB for 500 steps with seq_length=4096 — much faster than equivalent dense 35B because only ~3B parameters are active per token (MoE routing). On A100 40GB with 4-bit QLoRA, allow ~4-6 hours. Watch the loss curve in the logs.

In [ ]:
trainer.train()

## 7. Save LoRA adapter

Two outputs: the LoRA-only adapter (small, ~200-500 MB) and the merged weights (full 64 GB, only if you need a single artefact). For Scrapalot inference deployment, prefer the adapter + base model approach.

In [ ]:
ADAPTER_OUT = "/workspace/checkpoints/qwen36_35b_a3b_sft_lora"
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print(f"Adapter saved to {ADAPTER_OUT}")

## 8. Export to GGUF for Scrapalot local inference (optional)

Scrapalot's local inference stack (`scrapalot-chat/src/main/service/llm_inference.py`) loads GGUF via `llama-cpp-python`. To deploy the fine-tuned model there, convert and copy to `service/local_models/<model_name>.gguf`.

**Caveat**: GGUF conversion of the merged Qwen3.6-35B-A3B model takes ~30-60 minutes; the q4_k_m output is ~20 GB (smaller than dense 35B because GGUF stores expert weights efficiently, and only 3B are active at inference time). Note that not all GGUF runtimes have first-class Qwen3.6 MoE support yet — verify your target llama.cpp version with `llama-cli --list-vocab`. Consider running this step only when you're ready to deploy.

In [ ]:
# Uncomment when ready to convert:
# model.save_pretrained_gguf(
#     '/workspace/checkpoints/qwen36_35b_a3b_scrapalot_q4km',
#     tokenizer,
#     quantization_method = 'q4_k_m',
# )

## 9. Expert routing introspection — which experts naturally activate per collection?

**Goal**: discover whether Qwen3.6-35B-A3B's pretrained expert pool (256 experts, 8 routed + 1 shared per token) already shows topic-level specialisation that maps onto Scrapalot collections. If it does, subsequent training iterations can target only the *naturally-activating* experts per collection (selective LoRA), or fine-tune the router to push collection queries toward known-good expert subsets.

**Method**:
1. Walk `model.named_modules()` and discover router/gate `Linear` layers (heuristic: `out_features ∈ {64, 128, 256, 512}` and name ending in `.gate` or `.router`).
2. Attach PyTorch forward hooks that capture top-K=8 expert IDs per token per layer.
3. Run N=20 sample queries per Scrapalot collection (questions from the dataset loaded in Section 4).
4. Aggregate `(collection, layer) → Counter(expert_id)`.
5. Compute the **distinctiveness ratio** for each (collection, expert): `freq_in_col / global_freq`. Ratios >> 1.0 mean "this expert disproportionately activates for this collection" — that's what we hope to see if MoE-per-collection has any natural foothold.
6. Save the full counters to `expert_routing_introspection.json` for offline analysis.

**Caveat**: Qwen3.6 uses Gated DeltaNet + Gated Attention layers, both non-standard. The router-module discovery is heuristic — if zero routers are found, inspect `model.named_modules()` manually and broaden the regex. Captures via forward hook are independent of any `output_router_logits` flag, so the cell works regardless of which transformers version is installed.

**Run mode**: works on both the base model (load Section 2 only, skip Sections 3-7, then run this cell) and the SFT-trained adapter (run all sections through 7, then this). Comparing the two reveals how SFT shifted expert preferences — a clean way to validate that fine-tuning actually changed something.

**Runtime**: ~5-10 minutes for 20 queries × N collections on A100. Scales linearly with `SAMPLES_PER_COLLECTION`.

In [ ]:
"""Expert routing introspection — see markdown above for goals & method."""

import collections
from collections import defaultdict
import json
import re

import torch

# Step 1: Discover MoE router modules by name pattern.
router_modules: dict[str, torch.nn.Linear] = {}
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear) and re.search(r"\.(gate|router)$", name) and module.out_features in (64, 128, 256, 512):
        router_modules[name] = module

print(f"Discovered {len(router_modules)} router modules:")
for name, m in list(router_modules.items())[:5]:
    print(f"  {name}  → out_features={m.out_features}")
if len(router_modules) > 5:
    print(f"  ... and {len(router_modules) - 5} more")
if not router_modules:
    raise RuntimeError(
        "No router modules discovered. Inspect model.named_modules() and broaden the regex. "
        "Qwen3.6 may use a non-standard naming convention (Gated DeltaNet / Gated Attention)."
    )

NUM_EXPERTS = list(router_modules.values())[0].out_features
TOP_K = 8  # Qwen3.6-35B-A3B: 8 routed + 1 shared per token. We capture only the 8 routed.
print(f"NUM_EXPERTS = {NUM_EXPERTS},  TOP_K = {TOP_K}")

# Step 2: Attach forward hooks that capture top-K expert IDs per token per layer.
activation_counters: dict[tuple[str, str], collections.Counter] = defaultdict(collections.Counter)
current_collection = {"value": None}


def make_router_hook(layer_name: str):
    def hook(_module, _input, output):
        # output: [batch, seq_len, num_experts] router logits
        col = current_collection["value"]
        if col is None:
            return
        topk_ids = torch.topk(output, k=TOP_K, dim=-1).indices  # [batch, seq, k]
        activation_counters[(col, layer_name)].update(topk_ids.flatten().tolist())

    return hook


hook_handles = [mod.register_forward_hook(make_router_hook(n)) for n, mod in router_modules.items()]
print(f"Attached {len(hook_handles)} hooks.")

# Step 3: Run sample queries per collection.
SAMPLES_PER_COLLECTION = 20

by_collection = defaultdict(list)
for r in records:  # `records` was built in Section 4
    by_collection[r["collection"]].append(r["question"])

print(f"\nCollections in dataset: {list(by_collection.keys())}")

FastLanguageModel.for_inference(model)
for col, questions in by_collection.items():
    current_collection["value"] = col
    sampled = questions[:SAMPLES_PER_COLLECTION]
    for q in sampled:
        inputs = tokenizer(q, return_tensors="pt", truncation=True, max_length=512).to("cuda")
        with torch.no_grad():
            _ = model(**inputs, use_cache=False)
    print(f"  {col}: {len(sampled)} queries processed")

current_collection["value"] = None
for h in hook_handles:
    h.remove()

# Step 4: Aggregate per-collection counts across all layers.
per_collection_total: dict[str, collections.Counter] = defaultdict(collections.Counter)
for (col, _layer), counter in activation_counters.items():
    per_collection_total[col].update(counter)

global_counter = collections.Counter()
for c in per_collection_total.values():
    global_counter.update(c)

# Step 5: Compute distinctiveness ratio per (collection, expert).
print("\n=== Top 10 most distinctive experts per collection ===")
for col, c in per_collection_total.items():
    total_col = sum(c.values()) or 1
    total_global = sum(global_counter.values()) or 1
    scored = [(eid, (c[eid] / total_col) / max(global_counter[eid] / total_global, 1e-9)) for eid in c]
    scored.sort(key=lambda x: -x[1])
    print(f"\n  {col}:")
    print(f"    {'expert_id':>10}   {'distinctiveness':>15}   {'count':>10}")
    for eid, ratio in scored[:10]:
        print(f"    {eid:>10d}   {ratio:>13.2f}x   {c[eid]:>10d}")

# Step 6: Save full (collection, layer) × expert frequency to JSON.
OUT_PATH = "/workspace/checkpoints/expert_routing_introspection.json"
serializable = {f"{col}|{layer}": dict(counter) for (col, layer), counter in activation_counters.items()}
with open(OUT_PATH, "w") as f:
    json.dump(serializable, f, indent=2)
print(f"\nSaved full per-(collection, layer) expert frequency to {OUT_PATH}")

## 10. State-distribution + retention monitoring (per arXiv:2605.22731)

**Goal**: before the first real training run consumes RunPod credit, instrument the notebook so we can detect whether SFT shifted the model in a healthy way (mild regime, retention ≥ 0.98) or pushed it off-policy (stress regime, retention ≤ 0.85). Paper *"Post-Training is About States, Not Tokens"* (arXiv:2605.22731v1, 2026-05-21) shows that on Qwen3-0.6B, stress SFT loses 17 % of base accuracy on retention tasks while still appearing healthy on the training loss curve — **loss alone is not a reliable signal**.

**Two metrics, both light-weight, no new dependencies**:

1. **Retention ratio on a frozen multiple-choice probe set** — 20 general-knowledge MC questions (capitals, basic chem, history, arithmetic). Score base model vs post-trained on the same set. `retention_ratio = posttrain_acc / base_acc`. Targets:
   - `≥ 0.98` → SFT is in the mild regime, safe to ship.
   - `0.85 – 0.98` → moderate forgetting, consider reducing epochs / LR or switching to OPD.
   - `< 0.85` → stress regime (paper's Stress-SFT row landed at 0.83); switch to RL or OPD-from-base.

2. **Rollout-state MMD (RBF kernel on hashed bag-of-token features)** — generates 20 open-ended responses from the same prompt set with base vs post-train, hashes bag-of-token-ids to a 1024-dim vector, computes unbiased MMD² between the two distributions. Bigger MMD = more state drift. **Important caveat from the paper**: MMD alone is **insufficient** — identical MMD (0.01093 vs 0.01092) can correspond to vastly different retention (0.83 vs 0.95). Use MMD as a *secondary* signal, not the primary acceptance gate.

**Implementation**:
- Uses PEFT's `model.disable_adapter_layers()` / `enable_adapter_layers()` to toggle the LoRA delta — same model object, two behaviours. No second model load required.
- Probe sets are inline below — extend or replace with Scrapalot-domain questions before the first production run if you want to also gate on domain accuracy.
- Output saved to `/workspace/checkpoints/state_distribution_metrics.json` for later comparison across training iterations.

**Run order**: this cell runs **after** Section 7 (save LoRA adapter). It works on the in-memory `model` from earlier sections; no checkpoint reload needed.

**Runtime**: ~2-3 minutes total on A100 (40 generations × ~128 tokens each, twice — once with adapter, once without).

In [ ]:
"""State-distribution + retention monitoring — per arXiv:2605.22731.

Runs after Section 7 (LoRA adapter saved). Uses PEFT adapter toggling to compare
baseline (adapter disabled) vs post-trained (adapter enabled) on the same probe
set. No second model load, no new dependencies.
"""

import json
import math
import re

import torch

# ---------- Probe data (20 + 20). Replace with Scrapalot-domain probes for
# domain-specific gating; keep both to track general-knowledge retention. ----------

RETENTION_PROBES: list[tuple[str, dict[str, str], str]] = [
    ("What is the capital of France?", {"A": "London", "B": "Paris", "C": "Berlin", "D": "Madrid"}, "B"),
    ("Which gas do humans breathe in?", {"A": "Carbon dioxide", "B": "Hydrogen", "C": "Oxygen", "D": "Nitrogen"}, "C"),
    ("What is 7 multiplied by 8?", {"A": "54", "B": "56", "C": "64", "D": "48"}, "B"),
    ("Who wrote 'Romeo and Juliet'?", {"A": "Charles Dickens", "B": "Jane Austen", "C": "William Shakespeare", "D": "Mark Twain"}, "C"),
    ("What is the chemical symbol for gold?", {"A": "Au", "B": "Ag", "C": "Gd", "D": "Go"}, "A"),
    ("Which planet is closest to the sun?", {"A": "Venus", "B": "Earth", "C": "Mars", "D": "Mercury"}, "D"),
    ("What is the largest ocean on Earth?", {"A": "Atlantic", "B": "Indian", "C": "Pacific", "D": "Arctic"}, "C"),
    ("In what year did World War II end?", {"A": "1944", "B": "1945", "C": "1946", "D": "1939"}, "B"),
    ("What is the square root of 144?", {"A": "10", "B": "11", "C": "12", "D": "14"}, "C"),
    ("Which country gifted the Statue of Liberty to the United States?", {"A": "England", "B": "France", "C": "Spain", "D": "Italy"}, "B"),
    ("What is the boiling point of water at sea level in Celsius?", {"A": "90", "B": "100", "C": "110", "D": "120"}, "B"),
    ("How many continents are there?", {"A": "5", "B": "6", "C": "7", "D": "8"}, "C"),
    ("Who painted the Mona Lisa?", {"A": "Vincent van Gogh", "B": "Leonardo da Vinci", "C": "Pablo Picasso", "D": "Claude Monet"}, "B"),
    ("Which language is primarily spoken in Brazil?", {"A": "Spanish", "B": "English", "C": "Portuguese", "D": "French"}, "C"),
    ("What is the smallest prime number?", {"A": "0", "B": "1", "C": "2", "D": "3"}, "C"),
    ("Which organ pumps blood through the human body?", {"A": "Liver", "B": "Heart", "C": "Brain", "D": "Kidney"}, "B"),
    ("How many sides does a hexagon have?", {"A": "5", "B": "6", "C": "7", "D": "8"}, "B"),
    ("Which is the longest river in the world?", {"A": "Amazon", "B": "Nile", "C": "Yangtze", "D": "Mississippi"}, "B"),
    ("What gas makes up most of Earth's atmosphere?", {"A": "Oxygen", "B": "Carbon dioxide", "C": "Nitrogen", "D": "Hydrogen"}, "C"),
    (
        "Which scientist proposed the theory of general relativity?",
        {"A": "Isaac Newton", "B": "Galileo Galilei", "C": "Albert Einstein", "D": "Niels Bohr"},
        "C",
    ),
]

ROLLOUT_PROMPTS: list[str] = [
    "Explain photosynthesis in one paragraph.",
    "What are the main causes of inflation?",
    "Describe the structure of an atom.",
    "How does a bicycle work mechanically?",
    "What is the meaning of democracy?",
    "Summarise the plot of Hamlet in three sentences.",
    "Explain why the sky appears blue.",
    "What are the differences between bacteria and viruses?",
    "Describe how a refrigerator keeps food cold.",
    "Explain the concept of supply and demand.",
    "What is the role of mitochondria in a cell?",
    "Describe the water cycle.",
    "How does an internal combustion engine work?",
    "Explain the difference between AC and DC current.",
    "What were the main causes of World War I?",
    "Describe how vaccines train the immune system.",
    "Explain the concept of compound interest.",
    "What is the difference between weather and climate?",
    "Describe how GPS satellites determine location.",
    "Explain how a transistor amplifies a signal.",
]

# ---------- MMD with RBF kernel on hashed bag-of-tokens (paper's "lexical features") ----------

HASH_DIM = 1024


def featurize_text(text: str) -> torch.Tensor:
    """Bag-of-token-ids hashed into a fixed-dim, L2-normalised vector."""
    ids = tokenizer(text, truncation=True, max_length=512, add_special_tokens=False)["input_ids"]
    v = torch.zeros(HASH_DIM, dtype=torch.float32)
    for tid in ids:
        v[tid % HASH_DIM] += 1.0
    n = v.norm()
    return v / n if n > 0 else v


def mmd2_rbf(X: torch.Tensor, Y: torch.Tensor) -> float:
    """Unbiased MMD² with RBF kernel; sigma via the median-distance heuristic."""
    XY = torch.cat([X, Y])
    dists = torch.cdist(XY, XY)
    positive = dists[dists > 0]
    sigma = max(positive.median().item(), 1e-6) if positive.numel() > 0 else 1.0
    K = torch.exp(-dists.pow(2) / (2 * sigma**2))
    n, m = X.shape[0], Y.shape[0]
    Kxx = (K[:n, :n].sum() - K[:n, :n].diag().sum()) / max(n * (n - 1), 1)
    Kyy = (K[n:, n:].sum() - K[n:, n:].diag().sum()) / max(m * (m - 1), 1)
    Kxy = K[:n, n:].mean()
    return (Kxx + Kyy - 2 * Kxy).item()


# ---------- Probe runners ----------


def run_mc_probe() -> tuple[int, int]:
    """Multiple-choice retention probe. Greedy decode, parse first A-D letter."""
    correct = 0
    for q, opts, expected in RETENTION_PROBES:
        prompt = f"Question: {q}\nA) {opts['A']}\nB) {opts['B']}\nC) {opts['C']}\nD) {opts['D']}\nAnswer (one letter):"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        m = re.search(r"\b([ABCD])\b", decoded.upper())
        if m and m.group(1) == expected:
            correct += 1
    return correct, len(RETENTION_PROBES)


def run_rollout_probe() -> torch.Tensor:
    """Generate one rollout per prompt; return stacked feature matrix."""
    features = []
    for prompt in ROLLOUT_PROMPTS:
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to("cuda")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        features.append(featurize_text(text))
    return torch.stack(features)


# ---------- Driver ----------

FastLanguageModel.for_inference(model)

# Post-train (adapter enabled — current state after training + save).
posttrain_correct, n_probes = run_mc_probe()
posttrain_features = run_rollout_probe()

# Baseline (adapter disabled — PEFT toggles the LoRA delta off in place).
if not hasattr(model, "disable_adapter_layers"):
    raise RuntimeError(
        "model.disable_adapter_layers() not available — this cell expects a PEFT-wrapped model "
        "(produced by FastLanguageModel.get_peft_model in Section 3). Skip this cell on full-finetuned runs."
    )

model.disable_adapter_layers()
try:
    base_correct, _ = run_mc_probe()
    base_features = run_rollout_probe()
finally:
    model.enable_adapter_layers()

# Metrics.
base_acc = base_correct / n_probes
posttrain_acc = posttrain_correct / n_probes
retention_ratio = posttrain_acc / base_acc if base_acc > 0 else float("nan")
mmd2 = mmd2_rbf(base_features, posttrain_features)
mmd = math.sqrt(max(mmd2, 0.0))

metrics = {
    "paper_anchor": "arXiv:2605.22731#state-distribution",
    "n_retention_probes": n_probes,
    "base_correct": base_correct,
    "posttrain_correct": posttrain_correct,
    "base_accuracy": base_acc,
    "posttrain_accuracy": posttrain_acc,
    "retention_ratio": retention_ratio,
    "n_rollout_probes": len(ROLLOUT_PROMPTS),
    "hash_dim": HASH_DIM,
    "mmd2_rbf_lexical": mmd2,
    "mmd_rbf_lexical": mmd,
}

OUT_PATH = "/workspace/checkpoints/state_distribution_metrics.json"
with open(OUT_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print("\n=== State-distribution + retention (per arXiv:2605.22731) ===")
print(f"  Retention probes (n={n_probes}):")
print(f"    base_accuracy        = {base_acc:.4f}  ({base_correct}/{n_probes})")
print(f"    posttrain_accuracy   = {posttrain_acc:.4f}  ({posttrain_correct}/{n_probes})")
print(f"    retention_ratio      = {retention_ratio:.4f}")
print(f"  Rollout-state MMD  (n={len(ROLLOUT_PROMPTS)}, hashed bag-of-tokens, RBF kernel):")
print(f"    MMD                  = {mmd:.5f}   (MMD² = {mmd2:.6f})")
print("\n  Reference benchmarks (Qwen3-0.6B / GSM8K from paper Table 2):")
print("    Mild SFT       retention = 1.00, MMD = 0.00956")
print("    Stress SFT     retention = 0.83, MMD = 0.01093   ← substantial forgetting")
print("    OPD from SSFT  retention = 0.95, MMD = 0.01092")
print("    On-policy RL   retention = 0.99, MMD = 0.01098")
print("\n  Acceptance gate:")
print("    retention_ratio >= 0.98 → mild regime, ship.")
print("    retention_ratio  < 0.85 → switch to RL or OPD (paper §3.5).")
print("    NB: MMD alone does NOT predict retention; use only as secondary signal.")
print(f"\n  Saved to {OUT_PATH}")

## 11. Diversity metrics — TTR, bigram repetition, MAUVE-proxy (per arXiv:2605.22579)

**Goal**: measure the open-ended-generation diversity Hyperfitting is supposed to improve. Cheap-to-run baseline before any training, re-runs identically against any later checkpoint. Same prompt set as Section 10's `ROLLOUT_PROMPTS` (20 generic + room to extend).

**Three metrics, all greedy-decode (no temperature, no sampling)**:

1. **Type-Token Ratio (TTR)** — unique-tokens / total-tokens per response, averaged across prompts. Paper baseline ≈ 0.40 (greedy, original model); hyperfitted ≈ 0.68. Higher = more lexically diverse.
2. **Bigram repetition rate** — fraction of bigrams that have already appeared earlier in the same response, averaged. Paper baseline ≈ 0.59; hyperfitted ≈ 0.14. Lower = less repetition.
3. **MAUVE-proxy via inter-response Jaccard divergence** — average pairwise Jaccard distance between response token sets. Real MAUVE needs a reference corpus + heavy nearest-neighbour distance; we approximate with a cheap "responses-should-differ-from-each-other" signal. NOT a substitute for true MAUVE — flag as proxy.

**Adapter toggle**: identical pattern to Section 10. `model.disable_adapter_layers()` gives base behaviour, `enable_adapter_layers()` gives post-train. Same in-memory model, two measurements, no second load.

**When to run this cell**:
- **Before any training** — purely on the base model. The adapter toggle is a no-op (everything reports identical numbers). This gives the very first baseline.
- **After Section 7 saves an adapter** — full A/B with adapter toggle.
- **Across iterations** — diff the resulting `diversity_metrics.json` between checkpoints.

**Acceptance gate (paper-derived)**:
- TTR ≥ 0.55 (between baseline 0.40 and hyperfit 0.68) → meaningful diversity improvement.
- bigram_rep ≤ 0.30 (between baseline 0.59 and hyperfit 0.14) → meaningful repetition mitigation.
- Both gates together → ship the adapter. Either fails → consider running more epochs or switching to a different recipe.

**Runtime**: ~2-3 minutes on A100 (20 generations × 256 tokens each, twice).

**MoE caveat** (per audit, MEDIUM-risk path): paper tests dense models up to 8B. Qwen3.6-35B-A3B is a 48-layer MoE — full Hyperfitting may produce different dynamics than the paper's mechanistic analysis predicts. **Late-Stage LoRA** (paper §5, freeze early layers, LoRA only on final 5) is OUT OF SCOPE here: Unsloth's `FastLanguageModel.get_peft_model` does not expose PEFT's `layers_to_transform`, and freezing early MoE layers while routers stay active is an untested combination. Treat hyperfit results conservatively until we have a follow-up paper with MoE-specific evidence.

In [ ]:
"""Diversity metrics — TTR, bigram repetition, MAUVE-proxy.

Runs after Section 10. Uses the same ROLLOUT_PROMPTS list as the state-distribution
cell so the two metric files diff across the same prompt set. Adapter-toggle pattern
matches Section 10. No new dependencies.
"""

import json
import math

import torch

# ---------- Metric helpers ----------


def type_token_ratio(token_ids: list[int]) -> float:
    if not token_ids:
        return 0.0
    return len(set(token_ids)) / len(token_ids)


def bigram_repetition_rate(token_ids: list[int]) -> float:
    """Fraction of bigrams that already appeared earlier in the same sequence."""
    if len(token_ids) < 2:
        return 0.0
    # strict=True is safe — both slices have identical length N-1.
    bigrams = list(zip(token_ids[:-1], token_ids[1:], strict=True))
    seen: set[tuple[int, int]] = set()
    repeats = 0
    for bg in bigrams:
        if bg in seen:
            repeats += 1
        else:
            seen.add(bg)
    return repeats / len(bigrams)


def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)


def mauve_proxy_inter_response(responses_token_sets: list[set]) -> float:
    """Average pairwise Jaccard distance (1 - similarity). Higher = more diverse."""
    n = len(responses_token_sets)
    if n < 2:
        return 0.0
    pair_sum = 0.0
    pair_count = 0
    for i in range(n):
        for j in range(i + 1, n):
            pair_sum += 1.0 - jaccard(responses_token_sets[i], responses_token_sets[j])
            pair_count += 1
    return pair_sum / max(pair_count, 1)


# ---------- Greedy generation + metric collection ----------


def collect_diversity_metrics(prompts: list[str], max_new_tokens: int = 256) -> dict:
    """Greedy-decode N prompts, compute TTR / bigram_rep / inter-response MAUVE-proxy."""
    ttrs: list[float] = []
    bigram_rates: list[float] = []
    token_sets: list[set] = []
    sample_outputs: list[str] = []

    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        new_tokens = out[0][inputs["input_ids"].shape[1] :].tolist()
        ttrs.append(type_token_ratio(new_tokens))
        bigram_rates.append(bigram_repetition_rate(new_tokens))
        token_sets.append(set(new_tokens))
        sample_outputs.append(tokenizer.decode(new_tokens, skip_special_tokens=True))

    return {
        "n_prompts": len(prompts),
        "max_new_tokens": max_new_tokens,
        "ttr_mean": sum(ttrs) / max(len(ttrs), 1),
        "ttr_min": min(ttrs) if ttrs else 0.0,
        "ttr_max": max(ttrs) if ttrs else 0.0,
        "bigram_rep_mean": sum(bigram_rates) / max(len(bigram_rates), 1),
        "bigram_rep_min": min(bigram_rates) if bigram_rates else 0.0,
        "bigram_rep_max": max(bigram_rates) if bigram_rates else 0.0,
        "mauve_proxy_jaccard": mauve_proxy_inter_response(token_sets),
        "sample_first_response": sample_outputs[0] if sample_outputs else "",
    }


# ---------- Driver — same adapter-toggle pattern as Section 10 ----------

FastLanguageModel.for_inference(model)

# Post-train measurement (adapter enabled — current state after training + save).
posttrain = collect_diversity_metrics(ROLLOUT_PROMPTS)

# Baseline measurement (adapter disabled — PEFT toggles the LoRA delta off in place).
# Skip cleanly if the model is full-finetuned (no PEFT adapter to toggle).
if hasattr(model, "disable_adapter_layers"):
    model.disable_adapter_layers()
    try:
        base = collect_diversity_metrics(ROLLOUT_PROMPTS)
    finally:
        model.enable_adapter_layers()
else:
    base = None
    print("NOTE: model has no PEFT adapter — skipping baseline A/B (full-finetune path).")

# ---------- Acceptance gates (paper-derived, see Section 11 markdown) ----------

gate_ttr = posttrain["ttr_mean"] >= 0.55
gate_bigram = posttrain["bigram_rep_mean"] <= 0.30
gate_overall = gate_ttr and gate_bigram

metrics = {
    "paper_anchor": "arXiv:2605.22579#hyperfitting",
    "posttrain": posttrain,
    "base": base,
    "deltas": (
        {
            "ttr_delta": posttrain["ttr_mean"] - base["ttr_mean"],
            "bigram_rep_delta": posttrain["bigram_rep_mean"] - base["bigram_rep_mean"],
            "mauve_proxy_delta": posttrain["mauve_proxy_jaccard"] - base["mauve_proxy_jaccard"],
        }
        if base is not None
        else None
    ),
    "gates": {
        "ttr_mean_ge_0.55": gate_ttr,
        "bigram_rep_mean_le_0.30": gate_bigram,
        "overall_ship": gate_overall,
    },
    "paper_reference_numbers": {
        "baseline_ttr": 0.40,
        "hyperfit_ttr": 0.68,
        "baseline_bigram_rep": 0.59,
        "hyperfit_bigram_rep": 0.14,
        "hyperfit_mauve_real": 0.91,
    },
}

OUT_PATH = "/workspace/checkpoints/diversity_metrics.json"
with open(OUT_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print("\n=== Diversity metrics (per arXiv:2605.22579) ===")
print(f"  Prompts:  n = {posttrain['n_prompts']},  max_new_tokens = {posttrain['max_new_tokens']}")
print()
print(
    f"  POSTTRAIN  TTR mean = {posttrain['ttr_mean']:.4f}   bigram_rep = {posttrain['bigram_rep_mean']:.4f}   mauve_proxy = {posttrain['mauve_proxy_jaccard']:.4f}"
)
if base is not None:
    print(
        f"  BASE       TTR mean = {base['ttr_mean']:.4f}   bigram_rep = {base['bigram_rep_mean']:.4f}   mauve_proxy = {base['mauve_proxy_jaccard']:.4f}"
    )
    print(
        f"  Δ          TTR = {metrics['deltas']['ttr_delta']:+.4f}     bigram = {metrics['deltas']['bigram_rep_delta']:+.4f}     mauve = {metrics['deltas']['mauve_proxy_delta']:+.4f}"
    )
print()
print("  Paper reference (dense small models, narrative-continuation domain):")
print("    baseline TTR 0.40 → hyperfit TTR 0.68")
print("    baseline bigram_rep 0.59 → hyperfit bigram_rep 0.14")
print()
print(f"  Acceptance gates:  TTR ≥ 0.55 → {gate_ttr},   bigram_rep ≤ 0.30 → {gate_bigram}")
print(f"  Overall: {'SHIP' if gate_overall else 'DO NOT SHIP'} (both gates {'pass' if gate_overall else 'must pass'})")
print()
print(f"  Saved to {OUT_PATH}")

## Contribution log — arxiv papers iterating on this notebook

Each accepted training-related arxiv paper amends this notebook (no separate per-paper PRD). Each contribution lives behind a clearly-labelled markdown anchor heading so the tracking file (`scripts/scrapalot-competitive-analysis-scripts/analyzed_arxiv_papers.txt`) can reference it via `#anchor`.

Ordering rule: contributions stack chronologically (oldest first). Each contribution states **what changed** (cell numbers, new dependencies, new function definitions) and **how to measure the change** (what test/eval establishes the new baseline).

---

### Contribution from arXiv:2605.22817 (VPO — Vector Policy Optimization) {#vpo}

**Paper**: "Vector Policy Optimization: Training for Diversity Improves Test-Time Search" — arXiv:2605.22817v1 (2026-05-21).

**Status**: documented, NOT yet implemented as a runnable cell. The SFT baseline above must run first and capture its `best@k` baseline before VPO can be layered on top.

**What VPO adds**:
1. **Vector quality scores in the dataset**. `scripts/dataset_generator/qa_generator.py` must extend the Claude Q&A prompt + JSON schema to emit per-criterion scores instead of a single `quality_score` 1-5. Proposed schema:
   ```json
   {
     "factual_accuracy":  4,
     "completeness":      5,
     "citation_grounding":3,
     "clarity":           5
   }
   ```
   Then in this notebook, Cell 4 ("Dataset loading") changes to load this vector instead of the scalar quality_score.

2. **m=3 candidate generation per prompt**. The SFT trainer above produces one answer per prompt. VPO requires generating m=3 candidate completions in one autoregressive chain (shared prefix), evaluated together as a set.

3. **Set-level reward with Dirichlet-sampled weights**. Replace SFT loss with VPO's objective from paper §2 Eq. 1:
   ```
   R(S) = E_{w ~ Dir(α)} [ max_{y ∈ S} w^T r(x, y) ]
   ```
   - Sample K random weights `w^(k) ~ Dir(α)` per rollout group
   - For each `w^(k)`, compute `max` over the m candidate responses
   - Average over the K weight samples → set-level advantage
   - Apply uniformly to all tokens in the rollout (group-level)

4. **Drop-in GRPO replacement**. The Unsloth + TRL ecosystem has a `GRPOTrainer` (see `Qwen3_(4B)-GRPO.ipynb` reference notebook in the Unsloth notebooks index). VPO subclasses GRPOTrainer and overrides the advantage estimator. Estimated implementation effort: ~200 LOC of subclass + tests.

5. **Acceptance test (Phase 5 of the original arxiv triage)**: new `tests/integration/training/test_vpo_best_at_k.py` (to be added once VPO cell is runnable). Asserts VPO `best@30` exceeds SFT `best@30` by ≥5 pp on a held-out per-collection test set with `p < 0.05` (paired bootstrap, 1000 resamples), and diversity ratio (mean pairwise edit distance of m=3 candidates) ≥ 5× SFT baseline.

**Risks**:
- VPO trades pass@1 for pass@k — the SFT-then-VPO model will be **worse on first-answer use cases** (direct Q&A). Only worth applying when Scrapalot wraps the model in deep-research-style search (best-of-N, evolutionary refinement).
- Reward vector needs non-collinear dimensions. UltraFeedback-like near-collinear scores cause VPO to underperform scalar baselines.
- Compute: m=3 candidates means 3× training-time generation cost vs SFT. Budget accordingly.

---

### Contribution from arXiv:2605.22731 (Post-Training is About States, Not Tokens) {#state-distribution}

**Paper**: "Post-Training is About States, Not Tokens: A State Distribution View of SFT, RL, and On-Policy Distillation" — arXiv:2605.22731v1 (2026-05-21).

**Status**: implemented as **Section 10** above (runnable as-is after Section 7 saves the LoRA adapter).

**What changed**:
1. **New Section 10** (markdown + one code cell) — state-distribution + retention probe driver. Uses PEFT's `model.disable_adapter_layers()` / `enable_adapter_layers()` to compare the same in-memory model with and without the LoRA delta. No second model load, no new dependencies (only `torch`, already required by Unsloth).
2. **Inline probe sets** — 20 general-knowledge multiple-choice questions + 20 open-ended rollout prompts, defined as Python literals (`RETENTION_PROBES`, `ROLLOUT_PROMPTS`). Replace or augment with Scrapalot-domain questions before the first production training run if you want domain-specific retention gating.
3. **Helper functions** — `featurize_text` (bag-of-token-ids → 1024-dim hash-bucket, L2-normalised), `mmd2_rbf` (unbiased MMD² estimator with RBF kernel + median-distance heuristic for σ), `run_mc_probe` (greedy decode + regex-parse first A-D), `run_rollout_probe` (generate 128 tokens per prompt, hash to feature).
4. **Output artefact** — `/workspace/checkpoints/state_distribution_metrics.json` with: `base_accuracy`, `posttrain_accuracy`, `retention_ratio`, `mmd_rbf_lexical`, sample sizes, paper anchor. Diff across training iterations.

**Why it matters** (per paper §3.5): SFT loss can look healthy while retention quietly drops 17 pp (Qwen3-0.6B Stress-SFT: GSM8K 0.42, retention 0.83). MMD alone is also insufficient — Stress-SFT (MMD 0.01093, retention 0.83) and OPD-from-Stress-SFT (MMD 0.01092, retention 0.95) are indistinguishable on drift magnitude but 12 pp apart on retention. Two metrics needed, both cheap.

**Acceptance gate built into the cell output**:
- `retention_ratio >= 0.98` → mild regime, safe to ship the adapter.
- `0.85 <= retention_ratio < 0.98` → moderate forgetting; reduce LR or epochs, or switch to OPD.
- `retention_ratio < 0.85` → stress regime; switch to RL (GRPO) or OPD-from-base. Per paper Table 2 RL ran at retention 0.99.

**How to measure the change** (the paper's own framework):
- Baseline (today): retention probe has never been run; we have no prior retention numbers.
- Target: first SFT run on Scrapalot Q&A JSONL **must** produce `retention_ratio >= 0.85`. If it doesn't, the cell output tells you exactly which method to switch to before burning more RunPod credit.
- The metrics JSON is intended to be diffed across iterations (mild SFT vs stress SFT vs OPD vs RL — same probe sets, comparable numbers).

**Risks / limitations**:
- The probe set is small (20 MC + 20 rollouts) and generic. Treat as a smoke-test, not a definitive evaluation. A scaled-up version belongs in the future `scrapalot-evaluations` project (per `memory/feedback_evaluation_separate_project`).
- Hashed bag-of-tokens is a deliberately weak feature (cheap, no extra deps). Hidden-state MMD would be more sensitive but heavier; revisit if probes show suspicious near-zero MMD on a clearly-changed model.
- Adapter-disable via PEFT only reflects the **LoRA** delta. On full-finetuned runs (no LoRA) the cell raises and you'd need to reload the base model separately.

---

### Contribution from arXiv:2605.22579 (Beyond Temperature — Hyperfitting) {#hyperfitting}

**Paper**: "Beyond Temperature: Hyperfitting as a Late-Stage Geometric Expansion" — arXiv:2605.22579v1 (2026-05-21).

**Status**: implemented as a **runnable scaffold** across three cells. Default-OFF flag keeps the standard SFT baseline path intact; switch the flag to opt into the paper recipe.

**Counterintuitive observation the paper proves**: fine-tuning to near-zero training loss on a small (~2000) dataset *improves* open-ended-generation diversity under greedy decoding. Reported numbers: TTR 0.40 → 0.68 (+71%); bigram-repetition 0.59 → 0.14 (-76%); MAUVE 0.82-0.91, beating stochastic decoding tricks while staying deterministic. Mechanism (paper §4): the final transformer block undergoes a "geometric expansion" (effective dimensionality +80.8) that promotes deep-tail tokens past temperature-scaling's rank-preserving limit. Crucial control: temperature-scaling that matches the hyperfitted model's output entropy fails to recover the diversity gains (TTR 0.397 vs 0.684), proving the effect is not entropy sharpening.

**What changed**:
1. **Section 4 — dataset cell** (`HYPERFIT_MODE` flag, default False). When True, sorts records by `quality_score` desc and truncates to `HYPERFIT_TARGET_SAMPLES = 2000`. The standard ≥4 filter still runs first. `quality_score` is now preserved on each record (previously dropped) so the sort works.
2. **Section 5 — trainer config** (`if HYPERFIT_MODE: ... else: ...`). Hyperfit path: `num_train_epochs=260`, `max_steps=-1`, `weight_decay=0.0`, `lr_scheduler_type="constant"`. The paper's §2.1 recipe verbatim. Default path is unchanged.
3. **Section 11 — new markdown + code cell** with TTR / bigram_rep / MAUVE-proxy (inter-response Jaccard divergence) and adapter-toggle A/B. Output: `/workspace/checkpoints/diversity_metrics.json`. Uses the same `ROLLOUT_PROMPTS` list as Section 10 so the two metrics files diff against an identical prompt set.

**Acceptance gate (paper-derived)**:
- `TTR_mean ≥ 0.55` (halfway between baseline 0.40 and hyperfit 0.68)
- `bigram_rep_mean ≤ 0.30` (halfway between baseline 0.59 and hyperfit 0.14)
- Both must pass to ship the adapter.

**Phase 5 baseline strategy** (unlike #vpo and #state-distribution, this paper's metrics are *trivially measurable on the base model*):
- Run Section 11 BEFORE any training to establish today's baseline (adapter disabled is a no-op on the base model, so both A/B numbers will be identical).
- Re-run after each iteration; diff `diversity_metrics.json` files.

**Out of scope (paper §5 — Late-Stage LoRA)**:
- Paper proposes freezing layers 0..N-5 and applying LoRA only to the final 5 layers. Saves ~80% LoRA parameters, claims comparable diversity gain.
- **Blocked here** because: (a) Unsloth's `FastLanguageModel.get_peft_model()` does not expose PEFT's `layers_to_transform` argument — would need a custom PEFT injection (~50 LOC) or a patched Unsloth source; (b) freezing early MoE layers while routers keep routing on every layer is an untested combination. Qwen3.6-35B-A3B is 48-layer MoE; the paper's mechanistic analysis was on dense models up to 8B. Treat the MoE generalisation as a research question, not a recipe.

**Risks specific to our setup**:
- **MoE generalisation unproven**. Paper tested TinyLlama-1.1B, Qwen2.5-1.5B/3B, LLaMA-3.2-3B, Gemma-2-2B, LLaMA-3.1-8B, Qwen2.5-7B — all dense. The "final layer geometric expansion" was measured on a 22-layer dense architecture. Whether a 48-layer MoE shows the same dynamics is an open question.
- **Domain mismatch with paper's evaluation**. Paper measures on Fiction-Stories / WritingPrompts / AG News (narrative continuation). Our dataset is non-fiction Q&A from `dataset_generator`. The paper's own Limitations section notes TTR/bigram-repetition do NOT measure semantic coherence or factual accuracy — higher diversity can come at the cost of correctness. Pair Section 11 with Section 10's retention gate before shipping.
- **Compute cost**. Paper reports 20.7 h on RTX 4090 for full 260-epoch SFT on TinyLlama-1.1B. Our model is ~10× larger; even MoE-active-3B character won't get us under ~80-120 h on A100. Consider stopping at epoch 80-140 (paper §6 Table 8 shows TTR plateau already there).

**Next arxiv paper that contributes will get its own anchor below (e.g. `#dpo`, `#grpo`, etc.).**